# Data Extraction

A brief look at converting raw ScanImage tiffs to contiguous planar outputs.

See [mbo_utilities documentation](https://millerbrainobservatory.github.io/mbo_utilities/) for more details and examples.

In [2]:
import mbo_utilities as mbo
from pathlib import Path

In [3]:
help(mbo.imread)

Help on function imread in module mbo_utilities.lazy_array:

imread(inputs: 'str | Path | Sequence[str | Path]', **kwargs)
    Lazy load imaging data from supported file types.

    Currently supported file types:
    - .bin: Suite2p binary files (.bin + ops.npy)
    - .tif/.tiff: TIFF files (BigTIFF, OME-TIFF and raw ScanImage TIFFs)
    - .h5: HDF5 files
    - .zarr: Zarr v3

    Parameters
    ----------
    inputs : str, Path, ndarray, MboRawArray, or sequence of str/Path
        Input source. Can be:
        - Path to a file or directory
        - List/tuple of file paths
        - An existing lazy array
    **kwargs
        Extra keyword arguments passed to specific array readers.

    Returns
    -------
    array_like
        One of Suite2pArray, TiffArray, MboRawArray, MBOTiffArray, H5Array,
        or the input ndarray.

    Examples
    -------
    >>> from mbo_utilities import imread
    >>> arr = imread("/data/raw")  # directory with supported files, for full filename



## Raw File

In [4]:
raw_data_path = Path(r"D:\demo\raw")

In [5]:
!tree D:\demo\raw /F /A

Folder PATH listing
Volume serial number is 5C82-1870
D:\DEMO\RAW
    mk355_7_27_2025_180mw_right_m2_go_to_2x-mROI-880x1100um_220x550px_2um-px_14p00Hz_00001_00001_00001.tif
    mk355_7_27_2025_180mw_right_m2_go_to_2x-mROI-880x1100um_220x550px_2um-px_14p00Hz_00001_00001_00002.tif
    
No subfolders exist 



In [6]:
data = mbo.imread(raw_data_path)
data.shape

(1574, 14, 550, 440)

In [16]:
# roi=None to stitch mROI's
output_directory = raw_data_path.parent / "mrois"
mbo.imwrite(
    data,
    outpath=output_directory,
    planes=[1, 7, 14],
    roi=0,
    metadata={"example": "metadata-for-file"},
    overwrite=False,
    ext=".tiff",
    target_chunk_mb = 100,
    register_z = True
)

No s3d-job detected, preprocessing data.
Running Suite3D job...
    Loaded file into shared memory in 6.46 sec
    Workers completed in 8.47 sec
    Total time: 14.94 sec
Suite 3D init pass done in 134.0 seconds.
Preprocessed data saved to D:\demo\mrois\s3d-preprocessed
Registered z-planes, results saved to D:\demo\mrois\s3d-preprocessed.


Saving plane01_roi1.tiff:   0%|          | 0/4 [00:00<?, ?it/s]

Saving plane07_roi1.tiff:   0%|          | 0/4 [00:00<?, ?it/s]

Saving plane14_roi1.tiff:   0%|          | 0/4 [00:00<?, ?it/s]

Saving plane01_roi2.tiff:   0%|          | 0/4 [00:00<?, ?it/s]

Saving plane07_roi2.tiff:   0%|          | 0/4 [00:00<?, ?it/s]

Saving plane14_roi2.tiff:   0%|          | 0/4 [00:00<?, ?it/s]

You can also pre-stitch mROIs into a single image:

```python
output_directory = raw_data_path.parent / "mrois_stitched"

mbo.imwrite(
    data,
    outpath=output_directory,
    planes=None,  # all zplanes
    roi=None,  # stitched rois 
    metadata={"example": "metadata-for-file"},
    overwrite=False,
    ext=".zarr",
    target_chunk_mb=150,
    register_z=False
)
```